In [ ]:
# Safe setup block
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


In [ ]:
!pip install -q tensorflow keras-tuner torch torchvision torchaudio nlpaug augly wandb imbalanced-learn keras_cv seaborn matplotlib numpy pandas scikit-learn

In [ ]:
import sys
import tensorflow as tf
import torch

print(f"Python version: {sys.version}")
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")

print("\n--- GPU Availability ---")
print(f"TF GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"PyTorch GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"PyTorch Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# --- INLINED UTILS FUNCTION ---
import os
import random
import numpy as np
import tensorflow as tf
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for academic-grade visualizations
sns.set_theme(style="whitegrid", context="talk", palette="deep")

def set_seed(seed=42):
    """
    Ensures absolute reproducibility across all stochastic operations.
    Sets seeds for Python random, NumPy, TensorFlow, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # TensorFlow
    tf.random.set_seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"[Utils] Random seed globally set to {seed} for reproducibility.")

def plot_tf_history(history, title="Training History"):
    """
    Plots the accuracy and loss curves for a Keras/TensorFlow training history.
    """
    acc = history.history.get('accuracy', history.history.get('acc', []))
    val_acc = history.history.get('val_accuracy', history.history.get('val_acc', []))
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=20, fontweight='bold', y=1.05)

    # Accuracy Plot
    ax1.plot(epochs, acc, 'bo-', label='Training Acc', alpha=0.8, linewidth=2)
    ax1.plot(epochs, val_acc, 'ro-', label='Validation Acc', alpha=0.8, linewidth=2)
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    # Loss Plot
    ax2.plot(epochs, loss, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
    ax2.plot(epochs, val_loss, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
    ax2.set_title('Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.show()

def plot_torch_history(train_losses, val_losses, train_acc=None, val_acc=None, title="PyTorch Training History"):
    """
    Plots the loss (and optionally accuracy) curves for PyTorch manual training loops.
    """
    epochs = range(1, len(train_losses) + 1)
    
    if train_acc is not None and val_acc is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(title, fontsize=20, fontweight='bold', y=1.05)

        # Accuracy Plot
        ax1.plot(epochs, train_acc, 'bo-', label='Training Acc', alpha=0.8, linewidth=2)
        ax1.plot(epochs, val_acc, 'ro-', label='Validation Acc', alpha=0.8, linewidth=2)
        ax1.set_title('Accuracy')
        ax1.set_xlabel('Epochs')
        ax1.set_ylabel('Accuracy')
        ax1.legend()

        # Loss Plot
        ax2.plot(epochs, train_losses, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
        ax2.plot(epochs, val_losses, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
        ax2.set_title('Loss')
        ax2.set_xlabel('Epochs')
        ax2.set_ylabel('Loss')
        ax2.legend()
    else:
        plt.figure(figsize=(8, 6))
        plt.plot(epochs, train_losses, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
        plt.plot(epochs, val_losses, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
        plt.title(f"{title} - Loss", fontsize=16, fontweight='bold')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        
    plt.tight_layout()
    plt.show()

def compare_models_tf(histories_dict, metric='val_accuracy', title="Model Comparison"):
    """
    Overlays multiple Keras histories on a single plot for A/B testing comparison.
    histories_dict: dict of { 'Label': history_object }
    """
    plt.figure(figsize=(10, 7))
    plt.title(title, fontsize=18, fontweight='bold')
    
    for label, history in histories_dict.items():
        data = history.history.get(metric, history.history.get('val_acc', []))
        epochs = range(1, len(data) + 1)
        plt.plot(epochs, data, marker='o', label=label, linewidth=2, alpha=0.8)
        
    plt.xlabel('Epochs')
    plt.ylabel(metric.replace('_', ' ').title())
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()


# 01. Regularization Techniques: Combating Overfitting

## 1. Problem Definition
Deep neural networks are highly expressive models capable of fitting complex datasets perfectly. However, this high capacity often leads to **overfitting**, where the model memorizes the training data noise instead of learning the underlying true data distribution. Regularization techniques constrain the model's capacity or introduce noise during training to improve generalization on unseen data.

## 2. Theoretical Background

### L1 & L2 Regularization (Weight Decay)
Regularization adds a penalty term to the loss function $J(\theta)$:

$$ J_{reg}(\theta) = J(\theta) + \lambda \Omega(\theta) $$

- **L2 Regularization (Ridge)**: $\Omega(\theta) = \frac{1}{2} ||w||_2^2 = \frac{1}{2} \sum w_i^2$
  *Intuition*: Penalizes large weights, leading to smaller, more diffuse weight values. Gradients push weights towards zero proportionally to their magnitude.
- **L1 Regularization (Lasso)**: $\Omega(\theta) = ||w||_1 = \sum |w_i|$
  *Intuition*: Pushes weights exactly to zero, inducing **sparsity**. This acts as a built-in feature selector.

### Dropout
During training, Dropout randomly zeroes out neuron outputs with probability $p$. This prevents co-adaptation among neurons, forcing the network to learn redundant and more robust representations. At inference, neurons are scaled by $1-p$.

### Early Stopping
Early Stopping acts as implicit regularization. By halting training when validation performance degrades, we restrict the optimization trajectory, preventing the weights from growing too large and fitting the noise.

In [ ]:
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow.keras import layers, regularizers
import numpy as np
import sys

set_seed(42)

## 3. TensorFlow Implementation: A/B Testing Regularization
We implement a standard MLP with and without regularization on a synthetic dataset to empirically observe the bias-variance tradeoff.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=1500, n_features=200, n_informative=20, flip_y=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
def build_tf_model(reg_type=None, dropout=False):
    model = tf.keras.Sequential()
    reg = None
    if reg_type == 'l2': reg = regularizers.l2(0.01)
    elif reg_type == 'l1': reg = regularizers.l1(0.01)
        
    model.add(layers.Dense(128, activation='relu', input_shape=(200,), kernel_regularizer=reg))
    if dropout: model.add(layers.Dropout(0.5))
    model.add(layers.Dense(64, activation='relu', kernel_regularizer=reg))
    if dropout: model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train Baseline, L2, and Dropout Models
epochs = 100
hist_base = build_tf_model().fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0)
hist_l2 = build_tf_model(reg_type='l2').fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0)
hist_drop = build_tf_model(dropout=True).fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0)

In [ ]:
compare_models_tf({
    'Baseline (Overfits)': hist_base,
    'L2 Regularization': hist_l2,
    'Dropout': hist_drop
}, metric='val_accuracy', title='Generalization: Regularization A/B Test')

## 4. Critical Analysis & Conclusion
- **Baseline**: We see validation accuracy stall while training accuracy approaches 100%. This is textbook overfitting.
- **L2 Regularization**: Constrains weights smoothly, yielding a more stable validation curve.
- **Dropout**: Serves as a strong regularizer by enforcing redundancy. Often outperforms L2 on complex datasets.
- **Failure Cases**: L1 can aggressively zero-out informative features if the penalty is too high. Dropout can slow down convergence.